# Prompt Engineering Lab
**Client:** TechFlow Solutions  
**Goal:** Diagnose and fix an unreliable customer-service chatbot by iterating prompts through three versions (v1 → v2 → v3) across three tasks:
- **Task A** — Sentiment Analysis (classification)
- **Task B** — Product Description Generation (open-ended generation)
- **Task C** — Data Extraction (structured output)

**Scientific method used:**
1. Establish a baseline with zero-shot prompts (v1)
2. Measure failure rates over 5 / 10 / 15 runs
3. Improve with structured instructions (v2) and measure again
4. Improve with few-shot examples + Chain-of-Thought (v3) and measure again
5. Compare all versions quantitatively

---
## Cell 1 — Imports & Environment

In [ ]:
# Standard library
import os
import time
import json
from collections import Counter

# Third-party: load secrets from .env so the key never lives in the notebook
from dotenv import load_dotenv
load_dotenv()  # reads .env from the current working directory

# OpenAI client
from openai import OpenAI

# Initialise client — reads OPENAI_API_KEY from the environment automatically
client = OpenAI()

print("Environment loaded.")
# Quick sanity check: confirm the key is present (never print the key itself)
print("API key found:", bool(os.getenv("OPENAI_API_KEY")))

---
## Cell 2 — Experiment Configuration
All tunable constants live here so they can be changed in one place.

In [ ]:
# Model to use for all experiments
# gpt-4o-mini gives a good balance of cost and quality for this lab
MODEL = "gpt-4o-mini"

# Temperature controls randomness: 0 = deterministic, 1 = very random
# We use 0.7 deliberately for v1/v2 to expose inconsistency, then discuss
TEMPERATURE = 0.7

# Maximum tokens the model may return per call
MAX_TOKENS = 512

# Number of iterations for the systematic failure analysis
# The lab protocol requires 5 → 10 → 15 run checkpoints
RUNS_SMALL  = 5
RUNS_MEDIUM = 10
RUNS_LARGE  = 15

# Delay in seconds between consecutive API calls to avoid rate-limit errors
API_DELAY = 0.5

print(f"Config: model={MODEL}, temperature={TEMPERATURE}, max_tokens={MAX_TOKENS}")
print(f"Run checkpoints: {RUNS_SMALL} / {RUNS_MEDIUM} / {RUNS_LARGE}")

---
## Cell 3 — Helper Functions

In [ ]:
def call_openai(prompt, model=MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    """
    Send a single prompt to the OpenAI Chat API and return the response text.

    Parameters
    ----------
    prompt      : str   — The user message to send
    model       : str   — Which model to use (default from config)
    temperature : float — Randomness level 0-1 (default from config)
    max_tokens  : int   — Max response length (default from config)

    Returns
    -------
    str — The model's response, whitespace-stripped
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    # Extract the text content from the first (and only) choice
    return response.choices[0].message.content.strip()


def run_prompt_n_times(prompt, n=RUNS_LARGE, delay=API_DELAY, label="", **kwargs):
    """
    Call call_openai n times with the same prompt and collect all responses.
    A short delay between calls avoids rate-limit errors.

    Parameters
    ----------
    prompt : str  — Prompt to repeat
    n      : int  — How many times to call the API
    delay  : float— Seconds to wait between calls
    label  : str  — Optional name printed in the progress output
    **kwargs      — Forwarded to call_openai (e.g. temperature=0)

    Returns
    -------
    list[str] — All n responses in order
    """
    if label:
        print(f"\n--- {label} ({n} runs) ---")

    results = []
    for i in range(n):
        response = call_openai(prompt, **kwargs)
        results.append(response)
        # Truncate long responses in the progress log so output stays readable
        preview = response[:80] + "..." if len(response) > 80 else response
        print(f"  [{i+1:02d}/{n}] {preview}")
        time.sleep(delay)

    return results


def analyze_consistency(results, task_type="classification"):
    """
    Compute and print consistency metrics for a list of model responses.

    For classification tasks  → exact-match consistency (% of runs that match the mode)
    For generation tasks      → format-length consistency (% within ±20% of median length)
    For extraction tasks      → JSON validity rate (% of responses that parse as valid JSON)

    Parameters
    ----------
    results   : list[str] — All responses from run_prompt_n_times
    task_type : str       — "classification" | "generation" | "extraction"

    Returns
    -------
    dict — Metrics dictionary so results can be stored and compared later
    """
    total  = len(results)
    unique = len(set(results))

    # --- Classification metric: exact-match rate of the most common answer ---
    if task_type == "classification":
        most_common_val, most_common_cnt = Counter(results).most_common(1)[0]
        consistency_pct = round(most_common_cnt / total * 100, 1)
        print(f"  Total runs        : {total}")
        print(f"  Unique responses  : {unique}")
        print(f"  Most common answer: '{most_common_val[:60]}'")
        print(f"  Exact-match rate  : {most_common_cnt}/{total} = {consistency_pct}%")
        return {"consistency_pct": consistency_pct, "unique": unique, "most_common": most_common_val}

    # --- Generation metric: % of responses within ±20% of the median length ---
    elif task_type == "generation":
        lengths = [len(r.split()) for r in results]  # word counts
        lengths.sort()
        median_len = lengths[len(lengths) // 2]
        # A response is "consistent" if its word count is close to the median
        within_range = sum(1 for l in lengths if abs(l - median_len) / max(median_len, 1) <= 0.20)
        consistency_pct = round(within_range / total * 100, 1)
        print(f"  Total runs        : {total}")
        print(f"  Unique responses  : {unique}")
        print(f"  Word-count range  : {min(lengths)}–{max(lengths)} words (median {median_len})")
        print(f"  Length consistency: {within_range}/{total} within ±20% = {consistency_pct}%")
        return {"consistency_pct": consistency_pct, "unique": unique, "median_len": median_len}

    # --- Extraction metric: % of responses that are valid parseable JSON ---
    elif task_type == "extraction":
        valid_json_count = 0
        for r in results:
            # Try to find and parse any JSON block in the response
            try:
                # Strip markdown code fences if present
                cleaned = r.strip().strip("```json").strip("```").strip()
                # Find the first { ... } block
                start = cleaned.find("{")
                end   = cleaned.rfind("}") + 1
                json.loads(cleaned[start:end])
                valid_json_count += 1
            except Exception:
                pass  # response was not parseable JSON
        consistency_pct = round(valid_json_count / total * 100, 1)
        print(f"  Total runs        : {total}")
        print(f"  Unique responses  : {unique}")
        print(f"  Valid JSON rate   : {valid_json_count}/{total} = {consistency_pct}%")
        return {"consistency_pct": consistency_pct, "unique": unique, "valid_json": valid_json_count}

    else:
        raise ValueError(f"Unknown task_type '{task_type}'. Use classification / generation / extraction.")


# Master results store — keyed by [task][version][n_runs]
# Populated throughout the notebook so the final report cell can read it all at once
results_store = {
    "sentiment":   {},   # classification task
    "product":     {},   # generation task
    "extraction":  {},   # extraction task
}

print("Helper functions defined. Results store initialised.")

---
## Part 1 — Zero-Shot Baseline Prompts (v1)
**Hypothesis:** Minimal prompts with no format instructions will produce variable, inconsistent outputs.  
**Method:** Write the simplest possible prompt for each task and call the API once to confirm it returns *something*.

In [ ]:
# ── Task A: Sentiment Analysis ──────────────────────────────────────────────
# v1: no role, no format constraint, no examples — intentionally naive
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

# ── Task B: Product Description ─────────────────────────────────────────────
# v1: only product name and price — no length, tone, or structure requirements
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

# ── Task C: Data Extraction ──────────────────────────────────────────────────
# v1: asks to "extract information" without specifying what fields or what format
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

# Single test run for each — just to confirm the API responds before running at scale
print("=== Task A — Sentiment v1 (1 run) ===")
print(call_openai(sentiment_prompt_v1))

print("\n=== Task B — Product v1 (1 run) ===")
print(call_openai(product_prompt_v1))

print("\n=== Task C — Extraction v1 (1 run) ===")
print(call_openai(extraction_prompt_v1))

---
## Part 2 — Failure Analysis: 5 / 10 / 15 Runs
**Goal:** Measure how inconsistent v1 prompts are at scale.  
**Why three checkpoints?** 5 runs shows early variance; 10 confirms the trend; 15 is enough to compute a stable consistency percentage.

In [ ]:
# ── 5-Run Checkpoint ─────────────────────────────────────────────────────────
# Goal: get a first look at variance. Do all 5 responses look the same?

sent_v1_5  = run_prompt_n_times(sentiment_prompt_v1,  n=RUNS_SMALL, label="Sentiment v1")
prod_v1_5  = run_prompt_n_times(product_prompt_v1,    n=RUNS_SMALL, label="Product v1")
extr_v1_5  = run_prompt_n_times(extraction_prompt_v1, n=RUNS_SMALL, label="Extraction v1")

In [ ]:
# ── 10-Run Checkpoint ────────────────────────────────────────────────────────
# Goal: check whether new failure patterns emerge that didn't appear in 5 runs

sent_v1_10  = run_prompt_n_times(sentiment_prompt_v1,  n=RUNS_MEDIUM, label="Sentiment v1")
prod_v1_10  = run_prompt_n_times(product_prompt_v1,    n=RUNS_MEDIUM, label="Product v1")
extr_v1_10  = run_prompt_n_times(extraction_prompt_v1, n=RUNS_MEDIUM, label="Extraction v1")

In [ ]:
# ── 15-Run Checkpoint + Consistency Metrics ───────────────────────────────────
# This is the primary measurement used for all version comparisons

sent_v1_15  = run_prompt_n_times(sentiment_prompt_v1,  n=RUNS_LARGE, label="Sentiment v1")
prod_v1_15  = run_prompt_n_times(product_prompt_v1,    n=RUNS_LARGE, label="Product v1")
extr_v1_15  = run_prompt_n_times(extraction_prompt_v1, n=RUNS_LARGE, label="Extraction v1")

print("\n=== CONSISTENCY METRICS — v1 ===")

print("\nSentiment (classification — exact match rate):")
# For classification we expect one of three labels; exact match measures format discipline
results_store["sentiment"]["v1"] = analyze_consistency(sent_v1_15, task_type="classification")

print("\nProduct Description (generation — length consistency):")
# For generation, exact match is unfair; we measure whether the output length stays stable
results_store["product"]["v1"] = analyze_consistency(prod_v1_15, task_type="generation")

print("\nData Extraction (extraction — JSON validity rate):")
# For extraction the critical quality is whether the output can actually be parsed
results_store["extraction"]["v1"] = analyze_consistency(extr_v1_15, task_type="extraction")

### v1 Failure Analysis — Observations

Fill this table in after running the cell above:

| Task | Consistency % | Observed Failure Patterns |
|------|:------------:|---------------------------|
| Sentiment | _%_ | e.g. returns full sentences instead of one word, varies label capitalisation |
| Product Desc | _%_ | e.g. wildly different lengths, bullet points vs prose, missing price |
| Data Extraction | _%_ | e.g. plain text instead of JSON, different field names each run |

---
## Part 3 — Iteration 1: Clearer Instructions (v2)
**Changes made:**
- Added an explicit role to anchor the model's behaviour
- Defined exact output format and length constraints
- Removed all ambiguity about what fields to return

**Expected outcome:** Format consistency improves; content quality may still vary.

In [ ]:
# ── Task A v2: Sentiment ─────────────────────────────────────────────────────
# Key changes vs v1:
#   + Explicit role ("You are a sentiment classifier")
#   + Hard constraint: single word only, no punctuation, no explanation
sentiment_prompt_v2 = """
You are a sentiment classifier for a customer service platform.
Classify the customer message below as exactly one of: Positive, Negative, or Neutral.
Rules:
- Respond with only that single word
- No punctuation, no explanation, no extra words

Customer message: "I love this product! It's exactly what I needed."
"""

# ── Task B v2: Product Description ───────────────────────────────────────────
# Key changes vs v1:
#   + Fixed sentence count (3 sentences)
#   + Named tone and required content elements
#   + Explicit no-bullet-points rule
product_prompt_v2 = """
Write a product description for the item below.
Rules:
- Exactly 3 sentences — no more, no less
- Tone: professional and enthusiastic
- Sentence 1 must state the main benefit
- Sentence 2 must mention the price
- Sentence 3 must be a call to action
- No bullet points, no headers

Product: Wireless mouse, $29.99
"""

# ── Task C v2: Data Extraction ────────────────────────────────────────────────
# Key changes vs v1:
#   + Explicit list of fields to extract with allowed values
#   + Required output format: JSON only
#   + Explicit instruction: no extra text
extraction_prompt_v2 = """
Extract the following fields from the customer feedback and return them as a JSON object.

Required fields:
- order_id           (string: the number after #)
- order_date         (string: the date mentioned)
- delivery_speed     (string: must be exactly "fast", "slow", or "not_mentioned")
- packaging_condition (string: must be exactly "good", "damaged", or "not_mentioned")

Rules:
- Return only the JSON object
- No markdown, no code fences, no explanation

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

# Run all three v2 prompts 15 times and measure consistency
sent_v2_15  = run_prompt_n_times(sentiment_prompt_v2,  n=RUNS_LARGE, label="Sentiment v2")
prod_v2_15  = run_prompt_n_times(product_prompt_v2,    n=RUNS_LARGE, label="Product v2")
extr_v2_15  = run_prompt_n_times(extraction_prompt_v2, n=RUNS_LARGE, label="Extraction v2")

print("\n=== CONSISTENCY METRICS — v2 ===")

print("\nSentiment v2:")
results_store["sentiment"]["v2"] = analyze_consistency(sent_v2_15, task_type="classification")

print("\nProduct v2:")
results_store["product"]["v2"] = analyze_consistency(prod_v2_15, task_type="generation")

print("\nExtraction v2:")
results_store["extraction"]["v2"] = analyze_consistency(extr_v2_15, task_type="extraction")

---
## Part 4 — Iteration 2: Few-Shot + Chain-of-Thought (v3)
**Changes made:**
- **Sentiment:** 3 labelled examples (few-shot) to show the exact expected format
- **Product:** 2 worked examples (few-shot) demonstrating the 3-sentence structure
- **Extraction:** Explicit numbered reasoning steps before the final JSON (Chain-of-Thought)

**Why this works:** Few-shot examples show the model the exact output pattern to copy; CoT forces the model to reason carefully before committing to a structured answer.

In [ ]:
# ── Task A v3: Sentiment — Few-Shot ──────────────────────────────────────────
# Three examples covering all three classes teach the model the exact output token
sentiment_prompt_v3 = """
Classify each customer message as exactly one word: Positive, Negative, or Neutral.
Respond with only that single word — no punctuation, no explanation.

Message: "This is the worst purchase I've ever made."
Negative

Message: "It arrived on time."
Neutral

Message: "Absolutely amazing quality, exceeded all my expectations!"
Positive

Message: "I love this product! It's exactly what I needed."
"""

sent_v3_15 = run_prompt_n_times(sentiment_prompt_v3, n=RUNS_LARGE, label="Sentiment v3 (few-shot)")

print("\nSentiment v3 consistency:")
results_store["sentiment"]["v3"] = analyze_consistency(sent_v3_15, task_type="classification")

In [ ]:
# ── Task C v3: Extraction — Chain-of-Thought ─────────────────────────────────
# Numbered reasoning steps force the model to process each field before writing JSON
# The worked example anchors the format for the final output
extraction_prompt_v3 = """
Extract structured data from customer feedback. Follow these steps exactly:

Step 1: Find the order ID (the number after #).
Step 2: Find the order date.
Step 3: Decide delivery_speed — "fast", "slow", or "not_mentioned".
Step 4: Decide packaging_condition — "good", "damaged", or "not_mentioned".
Step 5: Write a JSON object with keys: order_id, order_date, delivery_speed, packaging_condition.

Worked example:
Feedback: "My order #99999 placed on Jan 1st came late and the box was fine."
Step 1: order_id = "99999"
Step 2: order_date = "January 1st"
Step 3: delivery_speed = "slow"
Step 4: packaging_condition = "good"
Step 5: {{"order_id": "99999", "order_date": "January 1st", "delivery_speed": "slow", "packaging_condition": "good"}}

Now process this feedback:
Feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

extr_v3_15 = run_prompt_n_times(extraction_prompt_v3, n=RUNS_LARGE, label="Extraction v3 (CoT)")

print("\nExtraction v3 consistency:")
results_store["extraction"]["v3"] = analyze_consistency(extr_v3_15, task_type="extraction")

In [ ]:
# ── Task B v3: Product Description — Few-Shot + Structure ────────────────────
# Two complete examples demonstrate the exact 3-sentence pattern
# The model learns sentence roles (benefit / feature+price / CTA) from the examples
product_prompt_v3 = """
Write a 3-sentence product description following this structure exactly:
  Sentence 1: Main benefit (what problem does it solve?)
  Sentence 2: Key feature + price
  Sentence 3: Call to action
Tone: professional and enthusiastic. No bullet points.

Example:
Product: Mechanical keyboard, $79.99
Description: Elevate your typing experience with tactile precision that makes every keystroke satisfying. This full-size mechanical keyboard delivers professional-grade performance at just $79.99. Upgrade your desk setup today and feel the difference from the very first key.

Example:
Product: USB-C hub, $34.99
Description: Expand your laptop's connectivity instantly with seven versatile ports in one compact device. This USB-C hub supports 4K HDMI, fast charging, and high-speed data transfer — all for only $34.99. Order now and simplify your workstation with one smart purchase.

Product: Wireless mouse, $29.99
Description:
"""

prod_v3_15 = run_prompt_n_times(product_prompt_v3, n=RUNS_LARGE, label="Product v3 (few-shot + structure)")

print("\nProduct v3 consistency:")
results_store["product"]["v3"] = analyze_consistency(prod_v3_15, task_type="generation")

---
## Part 5 — Final Comparison Report
**Metric used per task:**
- Sentiment   → exact-match rate (classification)
- Product Desc → length consistency ±20% of median (generation)
- Extraction  → valid JSON rate (extraction)

In [ ]:
# Pull consistency scores from the store and print a formatted comparison table

def score(task, version):
    # Returns the consistency % for a given task+version, or "N/A" if not yet run
    entry = results_store.get(task, {}).get(version, {})
    pct = entry.get("consistency_pct", None)
    return f"{pct}%" if pct is not None else "N/A"

print("=" * 62)
print(f"{'Task':<22} {'Metric':<28} {'v1':>5} {'v2':>5} {'v3':>5}")
print("-" * 62)
print(f"{'Sentiment':<22} {'exact-match rate':<28} {score('sentiment','v1'):>5} {score('sentiment','v2'):>5} {score('sentiment','v3'):>5}")
print(f"{'Product Description':<22} {'length consistency ±20%':<28} {score('product','v1'):>5} {score('product','v2'):>5} {score('product','v3'):>5}")
print(f"{'Data Extraction':<22} {'valid JSON rate':<28} {score('extraction','v1'):>5} {score('extraction','v2'):>5} {score('extraction','v3'):>5}")
print("=" * 62)
print("Target: v3 scores ≥ 80% across all tasks")

### Final Failure Analysis Table

| Task | v1 % | v2 % | v3 % | Main v1 Failure Patterns | Key Technique Applied |
|------|:----:|:----:|:----:|--------------------------|----------------------|
| Sentiment | | | | | Few-Shot |
| Product Desc | | | | | Few-Shot + Structure |
| Data Extraction | | | | | Chain-of-Thought |

---
## Part 5b — Task Variations (Prompt Tuning)
Test the final v3 prompts on different inputs to check they generalise, not just memorise.

In [ ]:
# Sentiment variations — one of each class to check generalisation
sentiment_test_cases = [
    ("Negative", "The shipping took forever and the item arrived broken."),
    ("Neutral",  "It works as described."),
    ("Neutral",  "Decent product for the price, nothing special."),
]

print("=== Sentiment v3 — variations ===")
for expected, message in sentiment_test_cases:
    # Swap the message into the v3 prompt template
    prompt = sentiment_prompt_v3.replace(
        "I love this product! It's exactly what I needed.", message
    )
    result = call_openai(prompt)
    # Show whether the model matched the expected label
    match = "✓" if result.strip().lower() == expected.lower() else "✗"
    print(f"  [{match}] Expected: {expected:<10} Got: {result:<15} | '{message[:50]}'")

In [ ]:
# Product variations — different products to confirm the 3-sentence structure holds
product_variations = [
    "Noise-cancelling headphones, $89.99",
    "Portable phone charger 20000mAh, $24.99",
]

print("=== Product v3 — variations ===")
for product in product_variations:
    prompt = product_prompt_v3.replace("Wireless mouse, $29.99", product)
    result = call_openai(prompt)
    sentence_count = result.count(".")
    print(f"\nProduct : {product}")
    print(f"Sentences detected: ~{sentence_count}")
    print(f"Output  : {result}")

In [ ]:
# Extraction variations — check JSON structure is preserved across different inputs
extraction_variations = [
    "Order #67890 from April 2nd arrived in perfect condition but took two weeks.",
    "Got my package (#11111) yesterday. No issues at all.",
]

print("=== Extraction v3 — variations ===")
for feedback in extraction_variations:
    prompt = extraction_prompt_v3.replace(
        "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.",
        feedback
    )
    result = call_openai(prompt)
    # Try to parse the JSON from the result to confirm it's valid
    try:
        cleaned = result.strip().strip("```json").strip("```").strip()
        start, end = cleaned.find("{"), cleaned.rfind("}") + 1
        parsed = json.loads(cleaned[start:end])
        status = "VALID JSON"
    except Exception:
        parsed = None
        status = "INVALID JSON"
    print(f"\nFeedback: {feedback}")
    print(f"Status  : {status}")
    print(f"Parsed  : {parsed}")